In [1]:
import os
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm

MODEL_NAME = "law-ai/InLegalBERT"
MAX_LEN = 512
HEAD_TOKENS = 128
TAIL_TOKENS = 382
BATCH_SIZE = 16
EPOCHS = 10
LR = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.05
MAX_GRAD_NORM = 1.0
HIDDEN_DROPOUT = 0.2
CLASSIFIER_DROPOUT = 0.3
PATIENCE = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class ILDCDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LEN, head_tokens=HEAD_TOKENS, tail_tokens=TAIL_TOKENS):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.head_tokens = head_tokens
        self.tail_tokens = tail_tokens
        self.cls_id = tokenizer.cls_token_id
        self.sep_id = tokenizer.sep_token_id
        self.pad_id = tokenizer.pad_token_id

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        ids = self.tokenizer.encode(self.texts[idx], add_special_tokens=False, truncation=False)
        budget = self.head_tokens + self.tail_tokens
        if len(ids) > budget:
            ids = ids[:self.head_tokens] + ids[-self.tail_tokens:]
        ids = [self.cls_id] + ids + [self.sep_id]
        input_ids = torch.full((self.max_len,), self.pad_id, dtype=torch.long)
        attention_mask = torch.zeros(self.max_len, dtype=torch.long)
        input_ids[:len(ids)] = torch.tensor(ids, dtype=torch.long)
        attention_mask[:len(ids)] = 1
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }


def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"]
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds = logits.argmax(dim=1).cpu()
            all_preds.extend(preds.tolist())
            all_labels.extend(labels.tolist())
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    return acc, f1, all_labels, all_preds


def train():
    train_df = pd.read_csv("/kaggle/input/datasets/kritiprsd/dataset/ildc_single_train.csv").drop_duplicates(subset="text").reset_index(drop=True)
    dev_df = pd.read_csv("/kaggle/input/datasets/kritiprsd/dataset/ildc_single_dev.csv")
    test_df = pd.read_csv("/kaggle/input/datasets/kritiprsd/dataset/ildc_test.csv")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=2)
    config.hidden_dropout_prob = HIDDEN_DROPOUT
    config.attention_probs_dropout_prob = HIDDEN_DROPOUT
    config.classifier_dropout = CLASSIFIER_DROPOUT
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, config=config).to(DEVICE)

    train_ds = ILDCDataset(train_df["text"].tolist(), train_df["label"].tolist(), tokenizer)
    dev_ds = ILDCDataset(dev_df["text"].tolist(), dev_df["label"].tolist(), tokenizer)
    test_ds = ILDCDataset(test_df["text"].tolist(), test_df["label"].tolist(), tokenizer)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    n_pos = (train_df["label"] == 1).sum()
    n_neg = (train_df["label"] == 0).sum()
    total = n_pos + n_neg
    class_weights = torch.tensor(
        [(total / (2 * n_neg)) ** 0.5, (total / (2 * n_pos)) ** 0.5],
        dtype=torch.float32,
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    total_steps = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)

    best_f1 = 0.0
    patience_counter = 0

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"epoch {epoch+1}"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        val_acc, val_f1, _, _ = evaluate(model, dev_loader)
        print(f"epoch {epoch+1} | train_loss {total_loss/len(train_loader):.4f} | val_acc {val_acc:.4f} | val_f1 {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            patience_counter = 0
            model.save_pretrained("mvp_inlegalbert_best")
            tokenizer.save_pretrained("mvp_inlegalbert_best")
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print("early stopping")
                break

    best_model = AutoModelForSequenceClassification.from_pretrained("mvp_inlegalbert_best").to(DEVICE)
    best_model.eval()
    test_acc, test_f1, test_labels, test_preds = evaluate(best_model, test_loader)
    print(f"\ntest_acc {test_acc:.4f} | test_f1 {test_f1:.4f}")
    print(classification_report(test_labels, test_preds, target_names=["rejected", "accepted"]))


if __name__ == "__main__":
    train()

config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/516 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/534M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (29 outputs left out)

epoch 10: 100%|██████████| 318/318 [09:13<00:00,  1.74s/it]


epoch 10 | train_loss 0.0746 | val_acc 0.6400 | val_f1 0.5883


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


test_acc 0.6381 | test_f1 0.5966
              precision    recall  f1-score   support

    rejected       0.61      0.74      0.67       755
    accepted       0.68      0.53      0.60       762

    accuracy                           0.64      1517
   macro avg       0.64      0.64      0.63      1517
weighted avg       0.65      0.64      0.63      1517



In [2]:
!pip install pdfplumber --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 85.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 102.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 96.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.3.0 which is incompatible.


In [3]:
import shutil
shutil.make_archive("/kaggle/working/mvp_inlegalbert_best", "zip", "/kaggle/working/mvp_inlegalbert_best")

'/kaggle/working/mvp_inlegalbert_best.zip'

In [4]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import pdfplumber

MODEL_DIR = "/kaggle/working/mvp_inlegalbert_best"
CASE_FILE_PATH = "/kaggle/input/datasets/kritiprsd/test-file/test.pdf"
MAX_LEN = 512
HEAD_TOKENS = 128
TAIL_TOKENS = 382
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {DEVICE}")


def extract_text_from_pdf(pdf_path):
    text_parts = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text_parts.append(page_text)
    full_text = "\n".join(text_parts)
    if not full_text.strip():
        raise ValueError(
            "No extractable text found. This PDF is likely scanned/image-based and needs OCR first."
        )
    return full_text


def load_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
    model.eval()
    return tokenizer, model


def prepare_input(text, tokenizer):
    ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
    budget = HEAD_TOKENS + TAIL_TOKENS
    if len(ids) > budget:
        ids = ids[:HEAD_TOKENS] + ids[-TAIL_TOKENS:]
    ids = [tokenizer.cls_token_id] + ids + [tokenizer.sep_token_id]
    input_ids = torch.full((1, MAX_LEN), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((1, MAX_LEN), dtype=torch.long)
    input_ids[0, :len(ids)] = torch.tensor(ids, dtype=torch.long)
    attention_mask[0, :len(ids)] = 1
    return input_ids.to(DEVICE), attention_mask.to(DEVICE)


def predict(text, tokenizer, model):
    input_ids, attention_mask = prepare_input(text, tokenizer)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs = F.softmax(logits, dim=1).squeeze(0).cpu()
    return {"rejected_prob": probs[0].item(), "accepted_prob": probs[1].item()}


if __name__ == "__main__":
    case_text = extract_text_from_pdf(CASE_FILE_PATH)

    tokenizer, model = load_model()
    result = predict(case_text, tokenizer, model)

    print(f"Chance of being cleared (accepted): {result['accepted_prob']*100:.1f}%")
    print(f"Chance of being rejected: {result['rejected_prob']*100:.1f}%")

using device: cuda


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (670 > 512). Running this sequence through the model will result in indexing errors


Chance of being cleared (accepted): 99.7%
Chance of being rejected: 0.3%
